# 해밀턴 경로로 연결한 은닉층: 설계·수학·성능·연산 효율 탐구

<a href="https://colab.research.google.com/github/hatnimi/claude_repo/blob/claude/relaxed-knuth-q3zflq/hamiltonian_nn/hamiltonian_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**탐구 질문**
1. 밀집층(dense layer)은 입력 뉴런 $n$개와 출력 뉴런 $n$개를 **모두** 잇는다($n^2$개 간선). 이 연결을 **해밀턴 경로**(모든 정점을 정확히 한 번씩 지나는 경로)로 제한하면 간선이 $2n-1$개로 줄어든다. 그래도 학습이 되는가?
2. 왜 하필 해밀턴 경로인가? 간선 수가 같은 **무작위 연결**, 간선이 더 적은 **완전 매칭(순열)** 과 비교하면 무엇이 다른가? → **라플라시안 행렬**로 설명
3. 이 층의 **역전파 수식**을 직접 유도하고, **야코비안**으로 검증할 수 있는가?
4. 경로(순열)를 사람이 정하지 않고 **학습**할 수 있는가? → 순열을 **이중확률행렬**로 완화하고 **싱크혼 알고리즘**(최적 운송)으로 학습
5. 이론 연산량은 $O(n^2) \to O(n)$으로 줄어든다. **실제 GPU 실행 시간**도 그만큼 줄어드는가? 아니라면 왜인가?

**실행 방법**: 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`를 고른 뒤, 위에서부터 차례로 실행한다. 전체 약 20~30분.

| 절 | 내용 | 수학 도구 |
|---|---|---|
| 1 | 해밀턴 경로로 층 만들기 | 그래프, 이분 그래프, 순열행렬 |
| 2 | 역전파 수식 유도와 검증 | 연쇄법칙, 야코비안 행렬 |
| 3 | 왜 해밀턴 경로인가 | 라플라시안 행렬, 고윳값, 대수적 연결도 |
| 4 | 실험 1: 성능 비교 | (FashionMNIST 분류) |
| 5 | 순열 → 이중확률행렬 | 버코프 정리, 최적 운송, 싱크혼 정리 |
| 6 | 실험 2: 경로를 학습하기 | 싱크혼 역전파, 헝가리안 알고리즘 |
| 7 | 실험 3: 연산 효율 이론 vs 실측 | 연산 복잡도, 메모리 대역폭, GPU 병렬성 |
| 8 | 결과 정리와 저장 | |

## 0. 준비
결과 그림과 표는 `outputs/` 폴더에 저장된다. 그림 글자는 한글 글꼴 문제를 피하려고 영어로 쓴다.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # 결과를 GitHub에 저장할 때만 필요하다. 저장소가 비공개라 clone이 실패해도 실험은 그대로 진행된다.
    REPO_DIR = "/content/hamiltonian_repo"
    if os.path.isdir(REPO_DIR + "/.git"):
        !git -C {REPO_DIR} pull -q
    else:
        !git clone -q -b claude/relaxed-knuth-q3zflq https://github.com/hatnimi/claude_repo.git {REPO_DIR}
    if os.path.isdir(REPO_DIR + "/hamiltonian_nn"):
        %cd {REPO_DIR}/hamiltonian_nn

In [ ]:
import math, time, json, platform, warnings
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
import networkx as nx
from scipy.optimize import linear_sum_assignment
from scipy.sparse.csgraph import laplacian, connected_components

QUICK = os.environ.get("HAM_QUICK") == "1"     # 동작 점검용 짧은 실행 (평소에는 False)
warnings.filterwarnings("ignore", message="Sparse CSR tensor support is in beta")
device = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"
print("PyTorch", torch.__version__, "| device:", device, "|", GPU_NAME, "|", platform.processor() or platform.machine())
if device == "cpu":
    print("⚠️ GPU가 없습니다. 메뉴 '런타임 → 런타임 유형 변경'에서 T4 GPU를 고르세요. (CPU로도 돌아가지만 7절의 GPU 분석은 빠집니다)")

OUT = "outputs"; os.makedirs(OUT, exist_ok=True)
RESULTS = {"device": device, "gpu": GPU_NAME, "torch": torch.__version__}

def seed_all(s):
    torch.manual_seed(s); np.random.seed(s)

def savefig(name):
    plt.tight_layout(); plt.savefig(f"{OUT}/{name}.png", dpi=130); plt.show()

## 1. 해밀턴 경로로 층 만들기

### 1-1. 층 = 이분 그래프
입력 뉴런 $x_1,\dots,x_n$과 출력 뉴런 $y_1,\dots,y_n$을 정점으로 보면, 한 층의 연결은 **이분 그래프**다.
밀집층은 모든 $x_j$와 $y_i$를 잇는 완전 이분 그래프 $K_{n,n}$이고 간선은 $n^2$개다. 가중치 $W_{ij}$가 간선 $(x_j, y_i)$ 하나에 대응한다.

### 1-2. $K_{n,n}$ 위의 해밀턴 경로
해밀턴 경로는 그래프의 **모든 정점을 정확히 한 번씩** 지나는 경로다.
이분 그래프에서 간선은 항상 $x$쪽과 $y$쪽을 오가므로, $2n$개 정점을 모두 지나는 경로는 반드시 $x, y, x, y, \dots$ 를 번갈아 지난다.
입력 쪽을 지나는 순서를 순열 $\pi$, 출력 쪽 순서를 순열 $\sigma$라 하면

$$x_{\pi(1)} \to y_{\sigma(1)} \to x_{\pi(2)} \to y_{\sigma(2)} \to \cdots \to x_{\pi(n)} \to y_{\sigma(n)}$$

간선은 $2n-1$개이고, 출력 $y_{\sigma(k)}$는 바로 앞뒤의 입력 $x_{\pi(k)}, x_{\pi(k+1)}$ 두 개와만 연결된다. 따라서

$$y_{\sigma(k)} = a_k\, x_{\pi(k)} + b_k\, x_{\pi(k+1)} + c_k \qquad (b_n = 0)$$

행렬로 쓰면 $W = P_\sigma^\top (D_a + D_b S) P_\pi$ 이다. ($P_\pi$: 순열행렬, $D_a=\mathrm{diag}(a)$, $S$: 한 칸 위로 미는 shift 행렬)
즉 **해밀턴 경로 층 = 순서를 바꾼 이중대각(bidiagonal) 행렬**이다.

### 1-3. 출력 순서 $\sigma$는 필요 없다
활성화 함수 $\phi$는 원소별로 작용하므로 순열과 교환된다: $\phi(P y) = P\phi(y)$.
층을 쌓으면 앞 층의 $P_\sigma^\top$와 다음 층의 $P_\pi$가 붙어서 하나의 순열 $P_{\pi'} = P_\pi P_\sigma^\top$가 된다.
그래서 일반성을 잃지 않고 $\sigma = $ 항등순열로 두고, **층마다 입력 순열 $\pi$ 하나만** 정하면 된다.

In [ ]:
def hamiltonian_path(perm_in, perm_out=None):
    # K_{n,n}에서 x_{π(1)}→y_{σ(1)}→x_{π(2)}→…→y_{σ(n)} 순서로 지나는 정점 목록
    n = len(perm_in)
    perm_out = list(range(n)) if perm_out is None else perm_out
    seq = []
    for k in range(n):
        seq += [("x", int(perm_in[k])), ("y", int(perm_out[k]))]
    return seq

def path_mask(perm):
    # 경로의 간선을 가중치 행렬 위치로: M[k, π(k)] = M[k, π(k+1)] = 1  (행 = 출력, 열 = 입력)
    n = len(perm); M = torch.zeros(n, n)
    M[torch.arange(n), perm] = 1
    M[torch.arange(n - 1), perm[1:]] = 1
    return M

def mask_to_graph(M):
    n_out, n_in = M.shape
    G = nx.Graph()
    G.add_nodes_from([("x", j) for j in range(n_in)] + [("y", i) for i in range(n_out)])
    G.add_edges_from((("x", int(j)), ("y", int(i))) for i, j in M.nonzero().tolist())
    return G

def is_hamiltonian_path(G, seq):
    # (1) 모든 정점을 정확히 한 번씩 지나고 (2) 이웃한 두 정점이 실제로 간선으로 이어져 있는가
    once = len(seq) == G.number_of_nodes() == len(set(seq))
    linked = all(G.has_edge(u, v) for u, v in zip(seq, seq[1:]))
    return once and linked

seed_all(0)
for n in [5, 64, 784]:
    perm = torch.randperm(n)
    M = path_mask(perm); G = mask_to_graph(M)
    print(f"n={n:4d}  간선 {G.number_of_edges():6d} (= 2n-1 = {2*n-1}),  밀집층 간선 {n*n:7d},  "
          f"해밀턴 경로? {is_hamiltonian_path(G, hamiltonian_path(perm.tolist()))},  "
          f"최대 차수 {max(d for _, d in G.degree())}")

In [ ]:
seed_all(1)
n = 6; perm = torch.randperm(n)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
pos = {("x", j): (j, 1) for j in range(n)} | {("y", i): (i, 0) for i in range(n)}
for a, (title, M) in zip(ax[:2], [("Dense layer: K_{6,6} (36 edges)", torch.ones(n, n)),
                                 ("Hamiltonian-path layer (2n-1 = 11 edges)", path_mask(perm))]):
    G = mask_to_graph(M)
    nx.draw(G, pos, ax=a, node_size=260, node_color=["#7aa6ff" if v[0] == "x" else "#ffb36b" for v in G],
            labels={v: f"{v[0]}{v[1]+1}" for v in G}, font_size=8, edge_color="#555")
    a.set_title(title)
seq = hamiltonian_path(perm.tolist())
nx.draw_networkx_edges(mask_to_graph(path_mask(perm)), pos, ax=ax[1], edgelist=list(zip(seq, seq[1:])),
                       edge_color="crimson", width=2.2)
n2 = 24; perm2 = torch.randperm(n2); M2 = path_mask(perm2)
ax[2].imshow(torch.cat([M2, torch.full((n2, 1), 0.5), M2[:, perm2]], 1), cmap="Greys")
ax[2].set_title("W mask (n=24): permuted | columns re-ordered by pi"); ax[2].axis("off")
savefig("1_hamiltonian_layer")
print("오른쪽 그림: 열을 π 순서로 다시 놓으면 대각선과 그 바로 위 대각선만 남는다 → 이중대각 행렬")

### 1-4. 구현: 행렬 곱 없이 '고르고, 밀고, 곱하기'
$u = P_\pi x$ (입력을 경로 순서로 재배열, `index_select`), $v = S u$ (한 칸 밀기), $y = a\odot u + b\odot v + c$.
$n\times n$ 행렬을 만들지 않으므로 파라미터는 $3n$개, 곱셈은 표본당 약 $2n$번이다.
역전파는 2절에서 직접 유도한 식으로 구현한 `HamiltonianFn`을 사용한다.

In [ ]:
class HamiltonianLinear(nn.Module):
    # y_k = a_k·x_{π(k)} + b_k·x_{π(k+1)} + c_k  (b_n = 0)  —  K_{n,n} 위의 해밀턴 경로 2n-1개 간선
    def __init__(self, n, perm=None, custom_backward=True):
        super().__init__()
        perm = torch.randperm(n) if perm is None else torch.as_tensor(perm)
        self.register_buffer("perm", perm.long())
        self.register_buffer("inv_perm", torch.argsort(perm).long())
        bmask = torch.ones(n); bmask[-1] = 0                     # 마지막 출력은 다음 입력이 없다
        self.register_buffer("bmask", bmask)
        # 입력이 2개(fan-in=2)이므로 He 초기화: Var = 2 / fan_in = 1
        self.a = nn.Parameter(torch.randn(n)); self.b = nn.Parameter(torch.randn(n))
        self.c = nn.Parameter(torch.zeros(n))
        self.custom_backward = custom_backward

    def forward(self, x):
        if self.custom_backward:
            return HamiltonianFn.apply(x, self.a, self.b * self.bmask, self.c, self.perm, self.inv_perm)
        u = x.index_select(1, self.perm)
        v = F.pad(u[:, 1:], (0, 1))
        return self.a * u + self.b * self.bmask * v + self.c

    def dense_weight(self):
        # 같은 층을 n×n 밀집 행렬로 펼친 것 (검증·비교용)
        n = len(self.perm); W = torch.zeros(n, n, device=self.a.device, dtype=self.a.dtype)
        W[torch.arange(n), self.perm] = self.a.detach()
        W[torch.arange(n - 1), self.perm[1:]] += (self.b * self.bmask).detach()[:-1]
        return W

## 2. 역전파 수식 유도와 야코비안 검증

### 2-1. 유도
배치의 표본 하나에 대해 $u_k = x_{\pi(k)}$, $v_k = u_{k+1}$ ($v_n=0$), $y_k = a_k u_k + b_k v_k + c_k$.
손실 $L$에 대해 $g_k = \dfrac{\partial L}{\partial y_k}$가 위층에서 내려왔다고 하자. 연쇄법칙으로

$$\frac{\partial L}{\partial a_k} = \sum_{\text{batch}} g_k u_k,\qquad
\frac{\partial L}{\partial b_k} = \sum_{\text{batch}} g_k v_k,\qquad
\frac{\partial L}{\partial c_k} = \sum_{\text{batch}} g_k$$

$u_k$는 $y_k$(계수 $a_k$)와 $y_{k-1}$(계수 $b_{k-1}$, $v_{k-1}=u_k$이므로) 두 곳에 쓰인다. 따라서

$$\frac{\partial L}{\partial u_k} = a_k g_k + b_{k-1} g_{k-1}\quad(k\ge 2),\qquad \frac{\partial L}{\partial u_1} = a_1 g_1$$

마지막으로 $u = P_\pi x$ 이므로 $\dfrac{\partial L}{\partial x} = P_\pi^\top \dfrac{\partial L}{\partial u}$, 즉 $\dfrac{\partial L}{\partial x_{\pi(k)}} = \dfrac{\partial L}{\partial u_k}$ (역순열로 되돌리기).

### 2-2. 야코비안
$J = \dfrac{\partial y}{\partial x}$의 $(k, \pi(k))$ 성분은 $a_k$, $(k, \pi(k+1))$ 성분은 $b_k$, 나머지는 0이다. 즉 **야코비안이 곧 $W$** 이고 0이 아닌 성분이 $2n-1$개다.
역전파는 $\dfrac{\partial L}{\partial x} = J^\top g$를 계산하는 일이므로 순전파와 같은 $O(n)$ 연산으로 끝난다.
활성화를 거친 $h=\phi(y)$의 야코비안은 $D_{\phi'(y)}W$이다.

In [ ]:
class HamiltonianFn(torch.autograd.Function):
    # 순전파와 '직접 유도한' 역전파. PyTorch 자동미분 대신 이 식으로 기울기를 계산한다.
    @staticmethod
    def forward(ctx, x, a, b, c, perm, inv_perm):
        u = x.index_select(1, perm)
        v = F.pad(u[:, 1:], (0, 1))
        ctx.save_for_backward(u, v, a, b, inv_perm)
        return a * u + b * v + c

    @staticmethod
    def backward(ctx, g):
        u, v, a, b, inv_perm = ctx.saved_tensors
        ga = (g * u).sum(0)                    # ∂L/∂a_k = Σ g_k u_k
        gb = (g * v).sum(0)                    # ∂L/∂b_k = Σ g_k v_k
        gc = g.sum(0)                          # ∂L/∂c_k = Σ g_k
        gu = a * g                             # ∂L/∂u_k = a_k g_k
        gu[:, 1:] += (b * g)[:, :-1]           #          + b_{k-1} g_{k-1}
        gx = gu.index_select(1, inv_perm)      # ∂L/∂x = P_πᵀ ∂L/∂u
        return gx, ga, gb, gc, None, None

# (1) gradcheck: 수치 미분 (f(θ+ε) - f(θ-ε)) / 2ε 과 직접 유도한 기울기를 double 정밀도로 비교
seed_all(0)
n = 12
layer = HamiltonianLinear(n).double()
x = torch.randn(5, n, dtype=torch.double, requires_grad=True)
b_eff = (layer.b * layer.bmask).detach().requires_grad_()
ok = torch.autograd.gradcheck(lambda x, a, b, c: HamiltonianFn.apply(x, a, b, c, layer.perm, layer.inv_perm),
                              (x, layer.a.detach().requires_grad_(), b_eff, layer.c.detach().requires_grad_()))
print("gradcheck (수치 미분과 일치?):", ok)

# (2) 자동미분과 비교
layer_auto = HamiltonianLinear(n, perm=layer.perm, custom_backward=False).double()
layer_auto.load_state_dict(layer.state_dict())
x1 = torch.randn(7, n, dtype=torch.double, requires_grad=True); x2 = x1.detach().clone().requires_grad_()
(layer(x1) ** 2).sum().backward(); (layer_auto(x2) ** 2).sum().backward()
print("자동미분과의 최대 차이: dx", (x1.grad - x2.grad).abs().max().item(),
      " da", (layer.a.grad - layer_auto.a.grad).abs().max().item())

# (3) 야코비안 = W 인가?
J = torch.autograd.functional.jacobian(lambda z: layer(z[None])[0], torch.randn(n, dtype=torch.double))
W = layer.dense_weight()
print("야코비안 == W ?", torch.allclose(J, W), "| 0이 아닌 성분:", int((J != 0).sum()), "= 2n-1 =", 2 * n - 1)

### 2-3. 층을 쌓으면: 야코비안의 0 아닌 성분이 퍼져 나간다 (수용 영역)
$L$개 층을 쌓은 전체 야코비안은 $J = J_L \cdots J_2 J_1$이다. 각 $J_\ell$의 행마다 0 아닌 성분이 2개이므로,
출력 하나가 의존하는 입력 수(수용 영역)는 최대 $2^L$개다. 모든 입력이 모든 출력에 닿으려면 $L \ge \log_2 n$층이 필요하다.
(층마다 순열이 다르기 때문에 이렇게 퍼진다. 순열이 같거나 연결이 '매칭'뿐이면 퍼지지 않는다.)

In [ ]:
def receptive_field(masks):
    # 0/1 마스크들을 곱해서 '출력 i가 입력 j에 의존하는가'를 구한다
    R = torch.eye(masks[0].shape[1])
    out = []
    for M in masks:
        R = ((M @ R) > 0).float()
        out.append(R)
    return out

seed_all(0)
n = 32; Ls = 6
Rs = receptive_field([path_mask(torch.randperm(n)) for _ in range(Ls)])
fig, ax = plt.subplots(1, Ls, figsize=(15, 2.8))
for l, (a, R) in enumerate(zip(ax, Rs)):
    a.imshow(R, cmap="Greys"); a.set_title(f"L={l+1}: {R.sum(1).mean():.1f} inputs/output"); a.axis("off")
savefig("2_jacobian_fill_in")

n = 784; depth = range(1, 13)
seed_all(0)
rf_ham = [R.sum(1).mean().item() for R in receptive_field([path_mask(torch.randperm(n)) for _ in depth])]
seed_all(0)
rf_same = [R.sum(1).mean().item() for R in receptive_field([path_mask(torch.arange(n)) for _ in depth])]
plt.figure(figsize=(6, 3.5))
plt.semilogy(depth, rf_ham, "o-", label="Hamiltonian, new random pi per layer")
plt.semilogy(depth, rf_same, "s-", label="Hamiltonian, same pi every layer")
plt.semilogy(depth, [min(2 ** L, n) for L in depth], "k--", label="min(2^L, n)")
plt.axhline(n, color="gray", lw=0.8); plt.xlabel("number of layers L"); plt.ylabel("inputs reached per output")
plt.title(f"Receptive field growth (n={n}, log2 n = {math.log2(n):.1f})"); plt.legend(); savefig("2_receptive_field")
RESULTS["receptive_field_784"] = dict(zip(depth, rf_ham))

### 2-4. 야코비안 특잇값: 기울기가 잘 전달되는가
깊은 망에서 전체 야코비안의 특잇값이 너무 크거나 작으면 기울기가 폭발하거나 사라진다.
초기화 상태에서 밀집층 망과 해밀턴 층 망의 전체 야코비안 특잇값 분포를 비교한다.

In [ ]:
def end_to_end_jacobian(layers, n, act=F.gelu):
    def f(z):
        h = z[None]
        for l in layers:
            h = act(l(h))
        return h[0]
    return torch.autograd.functional.jacobian(f, torch.randn(n))

seed_all(0)
n, L = 256, 6
sv = {}
for name, make in [("Dense (He init)", lambda: nn.Linear(n, n)),
                   ("Hamiltonian (He init)", lambda: HamiltonianLinear(n))]:
    layers = [make() for _ in range(L)]
    for l in layers:
        if isinstance(l, nn.Linear):
            nn.init.kaiming_normal_(l.weight); nn.init.zeros_(l.bias)
    J = end_to_end_jacobian(layers, n)
    s = torch.linalg.svdvals(J)
    sv[name] = s
    print(f"{name:22s} 최대 특잇값 {s[0]:9.3f}  중앙값 {s.median():.2e}  0에 가까운(<1e-6) 비율 {(s < 1e-6).float().mean():.2f}")
plt.figure(figsize=(6, 3.5))
for k, s in sv.items():
    plt.semilogy(s.numpy(), label=k)
plt.xlabel("index"); plt.ylabel("singular value"); plt.title(f"End-to-end Jacobian spectrum (n={n}, L={L})")
plt.legend(); savefig("2_jacobian_singular_values")

## 3. 왜 해밀턴 경로인가: 라플라시안 행렬로 보기

그래프의 **라플라시안** $L = D - A$ ($D$: 차수 대각행렬, $A$: 인접행렬)는 연결 구조를 고윳값으로 요약한다.
- 고윳값은 모두 $\ge 0$이고, **0의 개수 = 연결 성분의 개수**이다.
- 두 번째로 작은 고윳값 $\lambda_2$ (**대수적 연결도**, Fiedler 값)가 0보다 크면 그래프가 연결되어 있다. 클수록 정보가 빨리 퍼진다.

같은 $2n$개 정점(입력 $n$ + 출력 $n$)에 대해 네 가지 연결을 비교한다.

| 연결 | 간선 수 | 예상 |
|---|---|---|
| 밀집층 $K_{n,n}$ | $n^2$ | 고윳값 $0, n(\times 2n-2), 2n$ → $\lambda_2 = n$ |
| 해밀턴 경로 | $2n-1$ | 길이 $2n$인 경로 그래프 $P_{2n}$ → $\mu_k = 2-2\cos\frac{k\pi}{2n}$, $\lambda_2>0$ (연결) |
| 무작위 연결 | $2n-1$ | 간선 수는 같지만 대부분 **끊어짐**, 고립된 뉴런 발생 |
| 완전 매칭(순열) | $n$ | $n$개 조각으로 분리 → 고윳값 0이 $n$개 |

**핵심**: 정점 $2n$개를 하나로 연결하려면 간선이 최소 $2n-1$개 필요하다(신장 트리).
해밀턴 경로는 이 **최소 간선 수로 연결**을 이루면서, **모든 뉴런의 차수가 1~2로 고르다**(놀거나 과부하된 뉴런이 없다).

In [ ]:
def layer_laplacian_eigs(M):
    n_out, n_in = M.shape
    A = np.zeros((n_in + n_out, n_in + n_out))
    A[:n_in, n_in:] = M.T.numpy(); A[n_in:, :n_in] = M.numpy()
    return np.linalg.eigvalsh(laplacian(A)), A

def random_mask(n, edges, g=None):
    idx = torch.randperm(n * n, generator=g)[:edges]
    M = torch.zeros(n * n); M[idx] = 1
    return M.view(n, n)

def matching_mask(n):
    M = torch.zeros(n, n); M[torch.arange(n), torch.randperm(n)] = 1
    return M

seed_all(0)
n = 64
masks = {"Dense K_{n,n}": torch.ones(n, n), "Hamiltonian path": path_mask(torch.randperm(n)),
         "Random (2n-1 edges)": random_mask(n, 2 * n - 1), "Matching (n edges)": matching_mask(n)}
rows = []
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for name, M in masks.items():
    ev, A = layer_laplacian_eigs(M)
    comps = connected_components(A, directed=False)[0]
    deg = A.sum(1)
    rows.append((name, int(M.sum()), comps, abs(ev[1]), int((deg == 0).sum())))
    ax[0].plot(ev, ".", ms=3, label=name)
for r in rows:
    print(f"{r[0]:22s} 간선 {r[1]:5d} | 연결 성분 {r[2]:3d} | λ2 = {r[3]:.5f} | 고립 뉴런 {r[4]}")

# 해밀턴 경로 층의 고윳값 = 경로 그래프 P_2n 의 공식과 일치하는가
ev_ham, _ = layer_laplacian_eigs(masks["Hamiltonian path"])
N = 2 * n; formula = np.sort(2 - 2 * np.cos(np.arange(N) * np.pi / N))
print("\n해밀턴 경로 층의 라플라시안 고윳값 == 2-2cos(kπ/2n) ?", np.allclose(ev_ham, formula),
      f"(λ2 = {formula[1]:.5f})")
ax[0].set_yscale("symlog", linthresh=1e-2); ax[0].set_title(f"Laplacian eigenvalues of one layer (n={n})")
ax[0].set_xlabel("index"); ax[0].legend(fontsize=8)
ax[1].plot(ev_ham, "o", ms=3, label="numeric"); ax[1].plot(formula, "-", label="2 - 2cos(k pi / 2n)")
ax[1].set_title("Hamiltonian layer = path graph P_2n"); ax[1].legend()
savefig("3_laplacian_one_layer")
RESULTS["laplacian_one_layer_n64"] = [dict(zip(["kind", "edges", "components", "lambda2", "isolated"], r)) for r in rows]

### 3-1. 여러 층을 쌓은 전체 망의 연결도
$L$층을 쌓은 망 전체($(L+1)n$개 정점)의 라플라시안을 보면, 해밀턴 층은 몇 층을 쌓아도 하나로 연결되어 있고,
무작위 연결은 고립된 뉴런과 끊어진 조각이 계속 남는다. 끊어진 뉴런은 학습할 수 없는 '죽은' 뉴런이다.

In [ ]:
def stacked_graph_stats(masks):
    n = masks[0].shape[0]; L = len(masks); N = (L + 1) * n
    A = np.zeros((N, N))
    for l, M in enumerate(masks):
        A[(l + 1) * n:(l + 2) * n, l * n:(l + 1) * n] = M.numpy()
    A = A + A.T
    ev = np.linalg.eigvalsh(laplacian(A))
    return connected_components(A, directed=False)[0], abs(ev[1]), int((A.sum(1) == 0).sum())

seed_all(0)
n = 64
print(" L | 해밀턴: 성분/λ2       | 무작위(2n-1): 성분/λ2/고립 | 매칭: 성분")
for L in [1, 2, 4, 8]:
    h = stacked_graph_stats([path_mask(torch.randperm(n)) for _ in range(L)])
    r = stacked_graph_stats([random_mask(n, 2 * n - 1) for _ in range(L)])
    m = stacked_graph_stats([matching_mask(n) for _ in range(L)])
    print(f"{L:2d} | {h[0]:3d} / {h[1]:.5f}      | {r[0]:4d} / {r[1]:.5f} / {r[2]:3d}     | {m[0]:3d}")

## 4. 실험 1: 성능 비교 (FashionMNIST)

**모델 구조**: 입력(784픽셀) → [은닉층 784→784, GELU] × $L$ → 출력층 784→10
- 은닉층 너비를 입력과 같은 $n=784$로 두어, **은닉층의 연결 방식만** 바꿔 비교한다. 출력층(10개 클래스)은 모두 밀집층이다.
- 기준선: 은닉층 없음(= 로지스틱 회귀)
- 대조군: **무작위 연결**(해밀턴 경로와 간선 수 같음), **완전 매칭**(간선 $n$개) — 마스크를 씌운 밀집 행렬로 구현

MNIST는 너무 쉬워서 차이가 잘 안 보이므로 FashionMNIST(옷 사진 10종)를 쓴다. 바꾸려면 `DATASET = "MNIST"`.
데이터 전체를 GPU 메모리에 올려 DataLoader 오버헤드 없이 학습한다.

In [ ]:
import torchvision
DATASET = "FashionMNIST"     # 또는 "MNIST"
EPOCHS = 1 if QUICK else 10
SEEDS = [0] if QUICK else [0, 1, 2]
BATCH, LR = 256, 2e-3

def load(name):
    D = getattr(torchvision.datasets, name)
    tr, te = D("data", train=True, download=True), D("data", train=False, download=True)
    f = lambda d: ((d.data.float() / 255 - 0.5) / 0.5).reshape(len(d.data), -1).to(device)
    return f(tr), tr.targets.to(device), f(te), te.targets.to(device)

Xtr, ytr, Xte, yte = load(DATASET)
if QUICK:
    Xtr, ytr = Xtr[:8000], ytr[:8000]
print(DATASET, Xtr.shape, Xte.shape)

In [ ]:
class MaskedLinear(nn.Module):
    # 대조군용: 밀집 행렬에 0/1 마스크를 곱한다 (연결은 줄지만 연산량은 밀집층과 같다)
    def __init__(self, mask):
        super().__init__()
        self.register_buffer("mask", mask.float())
        fan_in = mask.sum(1).clamp(min=1)
        self.weight = nn.Parameter(torch.randn(mask.shape) * (2 / fan_in)[:, None].sqrt())
        self.bias = nn.Parameter(torch.zeros(mask.shape[0]))
    def forward(self, x):
        return x @ (self.weight * self.mask).t() + self.bias
    def n_params(self):
        return int(self.mask.sum()) + self.bias.numel()

def hidden_layer(kind, n):
    if kind == "dense":    return nn.Linear(n, n)
    if kind == "ham":      return HamiltonianLinear(n)
    if kind == "random":   return MaskedLinear(random_mask(n, 2 * n - 1))
    if kind == "matching": return MaskedLinear(matching_mask(n))
    raise ValueError(kind)

def make_model(kind, L, n=784):
    layers = []
    for _ in range(L):
        layers += [hidden_layer(kind, n), nn.GELU()]
    return nn.Sequential(*layers, nn.Linear(n, 10))

def count_params(model):
    # 마스크로 0이 된 가중치는 세지 않는다 (실제로 쓰이는 연결 수)
    total = 0
    for m in model.modules():
        if isinstance(m, MaskedLinear): total += m.n_params()
        elif isinstance(m, (nn.Linear, HamiltonianLinear)): total += sum(p.numel() for p in m.parameters(recurse=False))
    return total

@torch.no_grad()
def accuracy(model, X, y):
    model.eval(); acc = sum((model(X[i:i + 5000]).argmax(1) == y[i:i + 5000]).sum().item() for i in range(0, len(X), 5000)) / len(X)
    model.train(); return acc

def sync():
    if device == "cuda": torch.cuda.synchronize()

def train(model, epochs=EPOCHS, lr=LR, seed=0, on_step=None, log=False, theta_lr=None):
    g = torch.Generator(device=device).manual_seed(seed)
    model.to(device)
    if theta_lr is None:
        opt = torch.optim.Adam(model.parameters(), lr)
    else:   # 6절: 싱크혼 점수 행렬 θ만 큰 학습률
        th = [p for k, p in model.named_parameters() if k.endswith("theta")]
        rest = [p for k, p in model.named_parameters() if not k.endswith("theta")]
        opt = torch.optim.Adam([{"params": rest}, {"params": th, "lr": theta_lr}], lr)
    N = len(Xtr); steps = epochs * math.ceil(N / BATCH); step = 0; ep_times = []
    for ep in range(epochs):
        sync(); t0 = time.perf_counter()
        perm = torch.randperm(N, device=device, generator=g)
        for i in range(0, N, BATCH):
            if on_step: on_step(step / steps)
            idx = perm[i:i + BATCH]
            loss = F.cross_entropy(model(Xtr[idx]), ytr[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); step += 1
        sync(); ep_times.append(time.perf_counter() - t0)
        if log: print(f"  epoch {ep+1}: loss {loss.item():.3f}, test acc {accuracy(model, Xte, yte):.4f}")
    return accuracy(model, Xte, yte), float(np.median(ep_times))

In [ ]:
CONFIGS = [("logistic (no hidden)", "dense", 0),
           ("Dense L=1", "dense", 1), ("Dense L=2", "dense", 2),
           ("Hamiltonian L=1", "ham", 1), ("Hamiltonian L=2", "ham", 2),
           ("Hamiltonian L=4", "ham", 4), ("Hamiltonian L=8", "ham", 8),
           ("Random 2n-1 L=2", "random", 2), ("Matching L=2", "matching", 2)]
exp1 = {}
for name, kind, L in CONFIGS:
    accs, times = [], []
    for s in SEEDS:
        seed_all(s)
        model = make_model(kind, L)
        acc, t = train(model, seed=s)
        accs.append(acc); times.append(t)
    exp1[name] = dict(kind=kind, L=L, acc_mean=float(np.mean(accs)), acc_std=float(np.std(accs)),
                      params=count_params(model), epoch_sec=float(np.mean(times)))
    r = exp1[name]
    print(f"{name:22s} acc {100*r['acc_mean']:.2f} ± {100*r['acc_std']:.2f} %  | params {r['params']:>9,d} | {r['epoch_sec']:.2f} s/epoch")
RESULTS["exp1"] = exp1

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name, r in exp1.items():
    c = {"dense": "C0", "ham": "C3", "random": "C2", "matching": "C7"}[r["kind"]]
    ax[0].errorbar(r["params"], 100 * r["acc_mean"], yerr=100 * r["acc_std"], fmt="o", color=c, capsize=3)
    ax[0].annotate(name, (r["params"], 100 * r["acc_mean"]), fontsize=7, xytext=(4, 3), textcoords="offset points")
ax[0].set_xscale("log"); ax[0].set_xlabel("trainable parameters (log)"); ax[0].set_ylabel("test accuracy (%)")
ax[0].set_title(f"{DATASET}: accuracy vs parameters")
hams = [(r["L"], r) for r in exp1.values() if r["kind"] == "ham"]
ax[1].errorbar([L for L, _ in hams], [100 * r["acc_mean"] for _, r in hams], yerr=[100 * r["acc_std"] for _, r in hams],
               fmt="o-", color="C3", capsize=3, label="Hamiltonian")
for name, ls in [("logistic (no hidden)", ":"), ("Dense L=2", "--")]:
    ax[1].axhline(100 * exp1[name]["acc_mean"], ls=ls, color="gray", label=name)
ax[1].set_xlabel("Hamiltonian layers L"); ax[1].set_ylabel("test accuracy (%)"); ax[1].set_title("Depth sweep")
ax[1].legend(); savefig("4_accuracy")

**읽는 법 (실행 후 직접 확인하고 탐구노트에 기록할 것)**
- 해밀턴 층이 은닉층 없는 모델보다 얼마나 좋은가? → 간선 $2n-1$개짜리 층도 비선형 특징을 만들어 낸다는 증거
- 밀집층과의 차이는 몇 %p이고, 파라미터는 몇 배 차이인가?
- 간선 수가 같은 무작위 연결, 간선이 더 적은 매칭과 비교하면? → 3절의 라플라시안 결과(연결 성분, 고립 뉴런)와 연결 지어 설명
- 층을 깊게 쌓으면 수용 영역은 넓어지지만(2-3절) 정확도는 꼭 좋아지지 않을 수 있다. 왜일까? (최적화의 어려움, 표현력 vs 학습 가능성)

## 5. 순열에서 이중확률행렬로: 최적 운송과 싱크혼

지금까지 순열 $\pi$는 **무작위로** 정했다. 더 좋은 경로를 **학습**하고 싶지만, 순열은 $n!$개 중 하나를 고르는 이산적인 선택이라 미분할 수 없다.

### 5-1. 버코프–폰 노이만 정리
**이중확률행렬**: 모든 성분 $\ge 0$, 각 행의 합 = 각 열의 합 = 1.
버코프 정리: 이중확률행렬 전체의 집합(버코프 다면체)은 **순열행렬들의 볼록 결합**이고, 그 꼭짓점이 정확히 순열행렬이다.
→ 순열(꼭짓점) 대신 이중확률행렬(다면체 내부)에서 **연속적으로** 움직이며 학습하다가, 마지막에 가장 가까운 꼭짓점으로 내려앉히면 된다.

### 5-2. 싱크혼 정리
성분이 모두 양수인 정사각행렬 $A$에 대해, 양의 대각행렬 $D_1, D_2$가 존재해서 $D_1 A D_2$는 이중확률행렬이 된다(스케일 배수를 빼면 유일).
그리고 **행을 합으로 나누고, 열을 합으로 나누기를 번갈아 반복**하면 그 행렬로 수렴한다.

### 5-3. 최적 운송과의 관계 (엔트로피 정규화)
비용 $C_{ij}$로 분포 $r$을 분포 $c$로 옮기는 계획 $P$를 찾는 문제에 엔트로피 항을 더하면

$$\min_{P\ge0,\ P\mathbf 1 = r,\ P^\top \mathbf 1 = c}\ \langle C, P\rangle - \varepsilon H(P),\qquad H(P) = -\sum_{ij}P_{ij}(\log P_{ij}-1)$$

라그랑주 승수 $f_i, g_j$를 붙여 $P_{ij}$로 미분하면 $C_{ij} + \varepsilon\log P_{ij} - f_i - g_j = 0$, 즉

$$P_{ij} = e^{f_i/\varepsilon}\, e^{-C_{ij}/\varepsilon}\, e^{g_j/\varepsilon} \quad\Longrightarrow\quad P = \mathrm{diag}(u)\,K\,\mathrm{diag}(v),\ \ K = e^{-C/\varepsilon}$$

이것이 싱크혼 정리의 $D_1 A D_2$ 꼴이다. 제약을 맞추는 $u, v$는 $u \leftarrow r / (Kv)$, $v \leftarrow c/(K^\top u)$를 번갈아 반복해 구한다.
$r=c=\mathbf 1$이면 $P$는 이중확률행렬이고, $\varepsilon \to 0$이면 $P$는 $\langle C,P\rangle$를 최소화하는 **순열행렬**(선형 할당 문제의 해)로 다가간다.
신경망에서는 학습 가능한 점수 행렬 $\theta$를 $-C$로, 온도 $\tau$를 $\varepsilon$으로 쓴다.

In [ ]:
def sinkhorn_ot(C, r, c, eps, max_iters=20000, tol=1e-10):
    # 로그 영역 싱크혼: 작은 eps에서도 exp가 넘치지 않게 log u (=f/eps), log v (=g/eps) 로 계산
    logK = -C / eps; f = torch.zeros_like(r); g = torch.zeros_like(c); err = []
    for _ in range(max_iters):
        f = torch.log(r) - torch.logsumexp(logK + g[None, :], 1)     # u ← r / (K v)
        g = torch.log(c) - torch.logsumexp(logK + f[:, None], 0)     # v ← c / (Kᵀ u)
        P = torch.exp(f[:, None] + logK + g[None, :])
        err.append((P.sum(1) - r).abs().sum().item())               # 열 합은 방금 맞췄으니 행 합 오차만 본다
        if err[-1] < tol: break
    return P, err

m = 40; t = torch.linspace(0, 1, m, dtype=torch.double)
bump = lambda mu, s: torch.exp(-(t - mu) ** 2 / (2 * s ** 2))
r = bump(0.25, 0.08) + 0.5 * bump(0.7, 0.05); r /= r.sum()
c = bump(0.6, 0.12); c /= c.sum()
C = (t[:, None] - t[None, :]) ** 2
fig, ax = plt.subplots(1, 4, figsize=(15, 3.4))
ax[0].plot(t, r, label="source r"); ax[0].plot(t, c, label="target c"); ax[0].legend(); ax[0].set_title("two distributions")
for a, eps in zip(ax[1:], [0.05, 0.005, 0.0005]):
    P, err = sinkhorn_ot(C, r, c, eps)
    a.imshow(P, cmap="magma"); a.set_title(f"plan P, eps={eps}\ncost <C,P>={(C*P).sum():.4f}"); a.axis("off")
    print(f"eps={eps:<7} 반복 {len(err):5d}번에 수렴 (행 합 오차 {err[-1]:.1e}, 열 합 오차 {(P.sum(0) - c).abs().sum():.1e}), "
          f"운송 비용 {(C*P).sum():.5f}")
savefig("5_optimal_transport")
print("eps가 작을수록 운송 계획이 흐릿한 띠에서 날카로운 선(결정적인 대응)으로 바뀌고 비용이 줄지만, 수렴에 반복이 더 많이 필요하다.")

### 5-4. 싱크혼의 역전파를 직접 유도하기
로그 영역에서 '행 정규화' 한 번은 $\ell'_{ij} = \ell_{ij} - \log\sum_k e^{\ell_{ik}}$ 이다. $Q_{ij} = e^{\ell'_{ij}}$ (각 행의 합이 1)라 하면

$$\frac{\partial \ell'_{ij}}{\partial \ell_{ab}} = \delta_{ia}\left(\delta_{jb} - Q_{ab}\right)
\quad\Longrightarrow\quad
\frac{\partial L}{\partial \ell_{ab}} = G_{ab} - Q_{ab}\sum_j G_{aj}\qquad\left(G = \frac{\partial L}{\partial \ell'}\right)$$

열 정규화는 전치해서 같은 식을 쓴다. 싱크혼 전체는 이 두 단계를 반복한 것이므로, 역전파도 이 식을 거꾸로 반복하면 된다.
아래에서 이 식으로 `LogNormalize`의 backward를 직접 구현하고 gradcheck로 검증한 뒤, 이후 싱크혼 학습에 그대로 사용한다.

In [ ]:
class LogNormalize(torch.autograd.Function):
    # ℓ' = ℓ - logsumexp(ℓ, dim)  — 역전파: ∂L/∂ℓ = G - Q·ΣG  (직접 유도한 식)
    @staticmethod
    def forward(ctx, l, dim):
        out = l - torch.logsumexp(l, dim, keepdim=True)
        ctx.save_for_backward(out); ctx.dim = dim
        return out
    @staticmethod
    def backward(ctx, G):
        out, = ctx.saved_tensors
        return G - out.exp() * G.sum(ctx.dim, keepdim=True), None

def sinkhorn(theta, tau=1.0, iters=20):
    # 점수 행렬 θ → 이중확률행렬 (행 정규화와 열 정규화를 번갈아 iters번)
    l = theta / tau
    for _ in range(iters):
        l = LogNormalize.apply(l, 1)
        l = LogNormalize.apply(l, 0)
    return l.exp()

seed_all(0)
l0 = torch.randn(6, 6, dtype=torch.double, requires_grad=True)
print("LogNormalize gradcheck:", torch.autograd.gradcheck(lambda z: LogNormalize.apply(z, 1), (l0,)))
print("Sinkhorn 전체 gradcheck:", torch.autograd.gradcheck(lambda z: sinkhorn(z, 0.5, 5), (l0,)))

# 수렴 속도와 온도 τ → 0
theta = torch.randn(8, 8, dtype=torch.double)
row, col = linear_sum_assignment(-theta.numpy()); P_star = torch.zeros(8, 8, dtype=torch.double); P_star[row, col] = 1
fig, ax = plt.subplots(1, 5, figsize=(16, 3.2))
errs, l = [], theta.clone()
for _ in range(60):
    l = l - torch.logsumexp(l, 1, keepdim=True); l = l - torch.logsumexp(l, 0, keepdim=True)
    errs.append((l.exp().sum(1) - 1).abs().max().item())   # 열 정규화 직후 행 합이 1에서 얼마나 벗어났나
ax[0].semilogy(errs); ax[0].set_title("Sinkhorn: max |row sum - 1|"); ax[0].set_xlabel("iteration")
for a, tau in zip(ax[1:], [1.0, 0.3, 0.1, 0.03]):
    P = sinkhorn(theta, tau, 200)
    a.imshow(P, cmap="Blues", vmin=0, vmax=1)
    a.set_title(f"tau={tau}: |P - P*| = {(P - P_star).abs().sum():.2f}"); a.axis("off")
savefig("5_sinkhorn_temperature")
print("오른쪽으로 갈수록(τ→0) 이중확률행렬이 헝가리안 알고리즘의 답 P*(순열행렬)에 가까워진다.")

### 5-5. 버코프 정리 직접 확인: 이중확률행렬 = 순열행렬들의 볼록 결합
싱크혼이 만든 이중확률행렬을 실제로 순열행렬의 가중합 $\sum_i \lambda_i P_i$ ($\sum\lambda_i = 1$)로 분해해 본다.
(홀의 결혼 정리에 의해 이중확률행렬에는 항상 양수 성분만 지나는 순열이 존재하므로, 그 순열을 빼내는 일을 반복한다.)

In [ ]:
def birkhoff_decompose(D, tol=1e-9):
    D = D.clone().double(); terms = []
    while D.max() > tol:
        cost = (D <= tol).double() * 1e6 - D          # 0인 칸은 피하고, 큰 성분을 우선
        row, col = linear_sum_assignment(cost.numpy())
        lam = D[row, col].min().item()
        if lam <= tol: break
        P = torch.zeros_like(D); P[row, col] = 1
        terms.append((lam, P)); D -= lam * P
    return terms

seed_all(3)
D = sinkhorn(torch.randn(5, 5, dtype=torch.double), 1.0, 500)
terms = birkhoff_decompose(D)
recon = sum(l * P for l, P in terms)
print(f"순열행렬 {len(terms)}개로 분해, 가중치 합 = {sum(l for l, _ in terms):.6f}, 복원 오차 = {(recon - D).abs().max():.1e}")
print("가중치:", [round(l, 3) for l, _ in terms])

## 6. 실험 2: 해밀턴 경로를 학습하기

학습 중에는 순열 $P_\pi$ 대신 $\tilde P = \mathrm{Sinkhorn}(\theta/\tau)$ (이중확률행렬)를 쓴다: $u = \tilde P x$.
온도 $\tau$를 1에서 0.03까지 점점 낮추면 $\tilde P$가 순열행렬에 가까워진다.
학습이 끝나면 $\theta$에 대해 **헝가리안 알고리즘**(선형 할당 문제)으로 가장 가까운 순열 $\pi$를 고르고, 일반 `HamiltonianLinear`로 바꾼다(**경화**).
→ 학습할 때는 $O(n^2)$이지만, 추론할 때는 다시 $O(n)$이다.

### 6-1. 장난감 문제: 숨겨진 순열을 되찾을 수 있는가?
정답 층(teacher)이 만든 입력–출력 쌍만 보고, 학생 층이 정답 순열을 맞히는지 본다.
- (가) 정답이 **순열 하나** $y = P_\pi x$ 일 때
- (나) 정답이 **해밀턴 경로 층** $y = a\odot P_\pi x + b\odot S P_\pi x$ 일 때 (이웃 두 칸이 같은 순열을 공유)

In [ ]:
def sinkhorn_fast(theta, tau, iters):
    # 학습용: 위의 LogNormalize와 같은 계산을 PyTorch 기본 연산으로 (속도 비교용으로도 쓴다)
    l = theta / tau
    for _ in range(iters):
        l = l - torch.logsumexp(l, 1, keepdim=True)
        l = l - torch.logsumexp(l, 0, keepdim=True)
    return l.exp()

def recover_permutation(n, seed, task, steps=1500, use_custom=True):
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n, generator=g); a_t = torch.randn(n, generator=g); b_t = torch.randn(n, generator=g)
    bm = torch.ones(n); bm[-1] = 0
    def teacher(x):
        u = x[:, perm]
        return u if task == "perm" else a_t * u + b_t * bm * F.pad(u[:, 1:], (0, 1))
    theta = nn.Parameter(0.01 * torch.randn(n, n, generator=g))
    a = nn.Parameter(torch.ones(n)); b = nn.Parameter(torch.zeros(n))
    opt = torch.optim.Adam([theta] + ([a, b] if task == "ham" else []), lr=0.05)
    for t in range(steps):
        tau = 0.02 ** (t / steps)                       # 1 → 0.02
        x = torch.randn(256, n, generator=g)
        P = (sinkhorn if use_custom else sinkhorn_fast)(theta, tau, 20)
        u = x @ P.t()
        pred = u if task == "perm" else a * u + b * bm * F.pad(u[:, 1:], (0, 1))
        loss = F.mse_loss(pred, teacher(x)); opt.zero_grad(); loss.backward(); opt.step()
    _, col = linear_sum_assignment(-theta.detach().numpy())
    return (torch.as_tensor(col) == perm).float().mean().item(), loss.item()

toy = {}
for task in ["perm", "ham"]:
    res = [recover_permutation(16, s, task, steps=300 if QUICK else 1500) for s in range(2 if QUICK else 6)]
    toy[task] = res
    print(f"{task:5s} 순열 복원율: {[f'{r[0]:.2f}' for r in res]}  (평균 {np.mean([r[0] for r in res]):.2f}), "
          f"최종 손실 {[f'{r[1]:.3f}' for r in res]}")
RESULTS["toy_recovery"] = {k: [r[0] for r in v] for k, v in toy.items()}

**해석 포인트**: (가)는 볼록에 가까운 문제라 거의 항상 정답을 찾는다. (나)는 출력 $k$와 $k+1$이 같은 순열 행 $\pi(k+1)$을 공유하기 때문에
'반은 이쪽, 반은 저쪽'에 걸친 이중확률행렬 같은 **국소 최솟값**에 빠지기 쉽다. 해밀턴 경로를 고르는 문제는 본질적으로
**외판원 문제(TSP)** 와 비슷한 조합 최적화 문제라는 점과 연결해서 생각해 볼 수 있다.

### 6-2. FashionMNIST에서 경로 학습
은닉층 2개($L=2$)를 싱크혼 해밀턴 층으로 바꿔 `EPOCHS-1` epoch 학습 → 경화 → 순열을 고정한 채 1 epoch 더 학습(미세 조정).
비교 대상은 무작위 순열 해밀턴 층(L=2)을 같은 `EPOCHS` epoch 학습한 것이다.

설계하면서 정한 것 (점검 실험에서 시행착오로 찾음)
- $\theta$를 거의 0으로 시작하면 $\tilde P$가 모든 성분 $1/n$인 '평균 내기' 행렬이 되어 학습이 잘 안 된다 → **무작위 순열에서 출발**($\theta = P_{\text{rand}} + $ 작은 잡음)
- $\theta$는 순열을 바꿀 만큼 크게 움직여야 하므로 다른 파라미터보다 큰 학습률(0.05)을 준다
- 경화하면 정확도가 떨어진다($\tilde P$가 완전한 순열이 아니므로) → 순열을 고정하고 짧게 다시 학습해 회복되는지 본다

In [ ]:
class SinkhornHamiltonianLinear(nn.Module):
    def __init__(self, n, iters=10):
        super().__init__()
        theta = 0.01 * torch.randn(n, n)
        theta[torch.arange(n), torch.randperm(n)] += 1.0          # 무작위 순열에서 출발
        self.theta = nn.Parameter(theta)
        self.a = nn.Parameter(torch.randn(n)); self.b = nn.Parameter(torch.randn(n)); self.c = nn.Parameter(torch.zeros(n))
        bmask = torch.ones(n); bmask[-1] = 0; self.register_buffer("bmask", bmask)
        self.tau, self.iters = 1.0, iters
    def soft_perm(self):
        return sinkhorn_fast(self.theta, self.tau, self.iters)
    def forward(self, x):
        u = x @ self.soft_perm().t()                      # 학습 중: O(n²)
        return self.a * u + self.b * self.bmask * F.pad(u[:, 1:], (0, 1)) + self.c
    def harden(self):
        _, col = linear_sum_assignment(-self.theta.detach().cpu().numpy())
        h = HamiltonianLinear(len(col), perm=torch.as_tensor(col)).to(self.a.device)
        h.a.data.copy_(self.a.data); h.b.data.copy_(self.b.data); h.c.data.copy_(self.c.data)
        return h

def harden_model(model):
    return nn.Sequential(*[m.harden() if isinstance(m, SinkhornHamiltonianLinear) else m for m in model])

exp2 = {"learned_soft": [], "learned_hard": [], "learned_hard_ft": [], "random": [], "sharpness": []}
for s in SEEDS:
    seed_all(s)
    model = nn.Sequential(SinkhornHamiltonianLinear(784), nn.GELU(), SinkhornHamiltonianLinear(784), nn.GELU(), nn.Linear(784, 10))
    sh_layers = [m for m in model if isinstance(m, SinkhornHamiltonianLinear)]
    def anneal(frac):
        for m in sh_layers: m.tau = 1.0 * (0.03 / 1.0) ** frac      # τ: 1 → 0.03
    soft_acc, _ = train(model, epochs=max(EPOCHS - 1, 1), seed=s, on_step=anneal, theta_lr=0.05)
    with torch.no_grad():
        sharp = np.mean([m.soft_perm().max(1).values.mean().item() for m in sh_layers])
    hard = harden_model(model)
    hard_acc = accuracy(hard, Xte, yte)
    ft_acc, _ = train(hard, epochs=1, seed=s)                       # 순열 고정, a·b·c·출력층만 1 epoch
    seed_all(s); rand_acc, _ = train(make_model("ham", 2), seed=s)
    for k, v in zip(exp2, [soft_acc, hard_acc, ft_acc, rand_acc, sharp]): exp2[k].append(v)
    print(f"seed {s}: 학습(soft) {soft_acc:.4f} → 경화(hard) {hard_acc:.4f} → 미세조정 {ft_acc:.4f} | 무작위 순열 {rand_acc:.4f} | "
          f"P̃ 행별 최댓값 평균 {sharp:.3f} (1이면 완전한 순열)")
RESULTS["exp2"] = {k: float(np.mean(v)) for k, v in exp2.items()}
print({k: round(v, 4) for k, v in RESULTS["exp2"].items()})

## 7. 실험 3: 연산 효율 — 이론과 실측

### 7-1. 이론 (배치 $B$, 너비 $n$, fp32)
| | 밀집층 | 해밀턴 층 |
|---|---|---|
| 파라미터 | $n^2 + n$ | $3n$ |
| 곱셈·덧셈(FLOP) | $2Bn^2$ | 약 $4Bn$ |
| 메모리 읽기·쓰기 | $4(2Bn + n^2)$ 바이트 | 연산마다 $B\times n$ 텐서를 읽고 써서 약 $4\cdot 12\,Bn$ 바이트 |
| **산술 강도** (FLOP/바이트) | $B$와 $n$이 크면 수백 (계산이 병목) | 약 0.1 (메모리가 병목) |

GPU 시간은 대략 $\max\left(\dfrac{\text{FLOP}}{\text{최대 연산 속도}},\ \dfrac{\text{바이트}}{\text{메모리 대역폭}}\right) + \text{커널 실행 비용}\times\text{커널 수}$이다.
T4는 fp32 약 8.1 TFLOP/s, 메모리 약 320 GB/s이므로 산술 강도가 약 25보다 작으면 메모리 대역폭이 병목이다.
→ **예상**: $n$이 작을 때는 둘 다 커널 실행 비용(수 μs)에 묻혀 차이가 없거나, 커널을 여러 번 띄우는 해밀턴 층이 오히려 느리다.
$n$이 커지면 밀집층은 $n^2$으로, 해밀턴 층은 $n$으로 늘어나 결국 해밀턴 층이 빨라진다. 그 교차점이 어디인지 측정한다.

### 7-2. 측정 방법
- 같은 입력으로 10번 미리 실행(warm-up)한 뒤 여러 번 실행해 평균 시간을 잰다.
- GPU는 비동기로 동작하므로 `torch.cuda.Event`와 `synchronize()`로 실제 GPU 시간을 잰다.
- 비교: 밀집(`nn.Linear`, cuBLAS) / 마스크 밀집 / 해밀턴(`index_select` 등 기본 연산) / 해밀턴(`torch.compile`로 커널 융합) / 희소 행렬(CSR, cuSPARSE)

In [ ]:
PEAK = {"T4": (8.1e12, 320e9), "L4": (30.3e12, 300e9), "A100": (19.5e12, 1555e9), "V100": (15.7e12, 900e9)}
peak_flops, peak_bw = next((v for k, v in PEAK.items() if k in GPU_NAME), (8.1e12, 320e9))

def bench(fn, iters=None, warmup=10, dev=None):
    dev = dev or device
    with torch.no_grad():
        for _ in range(warmup): fn()
        if dev == "cuda":
            iters = iters or 50
            s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            torch.cuda.synchronize(); s.record()
            for _ in range(iters): fn()
            e.record(); torch.cuda.synchronize()
            return s.elapsed_time(e) / iters                    # ms
        iters = iters or 20
        t0 = time.perf_counter()
        for _ in range(iters): fn()
        return (time.perf_counter() - t0) / iters * 1e3

def ham_plain(layer):
    a, b, c, perm = layer.a, layer.b * layer.bmask, layer.c, layer.perm
    def f(x):
        u = x.index_select(1, perm)
        return a * u + b * F.pad(u[:, 1:], (0, 1)) + c
    return f

TRY_COMPILE = device == "cuda"          # torch.compile: 여러 연산을 커널 하나로 합친다 (첫 실행에 컴파일 시간이 걸림)
torch._dynamo.config.cache_size_limit = 64

def time_layers(n, B, compile_ok=TRY_COMPILE):
    x = torch.randn(B, n, device=device)
    dense = nn.Linear(n, n).to(device)
    ham = HamiltonianLinear(n).to(device)
    mask = path_mask(ham.perm.cpu()).to(device)
    W_csr = ham.dense_weight().to_sparse_csr()
    t = {"dense (matmul)": bench(lambda: dense(x)),
         "masked dense": bench(lambda: x @ (dense.weight * mask).t() + dense.bias),
         "hamiltonian (eager)": bench(lambda: ham_plain(ham)(x)),
         "sparse CSR": bench(lambda: (W_csr @ x.t()).t())}
    if compile_ok:
        try:
            fc = torch.compile(ham_plain(ham), dynamic=False)
            t["hamiltonian (compiled)"] = bench(lambda: fc(x))
        except Exception as ex:
            print("torch.compile 생략:", type(ex).__name__)
    return t

NS = [128, 256, 512] if QUICK else [128, 256, 512, 1024, 2048, 4096, 8192, 16384]
B = 512
timing = {}
for n in NS:
    timing[n] = time_layers(n, B)
    print(f"n={n:6d} | " + " | ".join(f"{k} {v:8.4f} ms" for k, v in timing[n].items()))
RESULTS["timing_forward_ms"] = {str(n): v for n, v in timing.items()}

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for k in timing[NS[0]]:
    ax[0].loglog(NS, [timing[n].get(k, np.nan) for n in NS], "o-", label=k)
ns = np.array(NS, dtype=float)
ax[0].loglog(ns, 2 * B * ns ** 2 / peak_flops * 1e3, "k--", lw=1, label="theory dense: 2Bn^2 / peak FLOPs")
ax[0].loglog(ns, 48 * B * ns / peak_bw * 1e3, "k:", lw=1, label="theory ham: ~48Bn bytes / bandwidth")
ax[0].set_xlabel("width n"); ax[0].set_ylabel("forward time (ms)"); ax[0].set_title(f"Measured vs theory (B={B}, {GPU_NAME})")
ax[0].legend(fontsize=7)
ratio = [timing[n]["dense (matmul)"] / timing[n]["hamiltonian (eager)"] for n in NS]
theo = [(2 * B * n * n) / (4 * B * n) for n in NS]
ax[1].loglog(NS, ratio, "o-", color="C3", label="measured speed-up (dense / ham eager)")
if "hamiltonian (compiled)" in timing[NS[0]]:
    ax[1].loglog(NS, [timing[n]["dense (matmul)"] / timing[n]["hamiltonian (compiled)"] for n in NS], "o-", color="C4",
                 label="measured speed-up (dense / ham compiled)")
ax[1].loglog(NS, theo, "k--", label="FLOP ratio n/2")
ax[1].axhline(1, color="gray", lw=0.8); ax[1].set_xlabel("width n"); ax[1].set_ylabel("speed-up"); ax[1].legend(fontsize=8)
ax[1].set_title("Theory (FLOP ratio) vs measured speed-up")
savefig("7_timing_vs_n")

print(f"{'n':>6} | {'dense GFLOP/s':>13} | {'dense 활용률':>9} | {'ham 유효 GB/s':>12} | {'측정 가속':>8} | {'FLOP 비':>8}")
for n, rr in zip(NS, ratio):
    gf = 2 * B * n * n / (timing[n]["dense (matmul)"] * 1e-3) / 1e9
    gb = 48 * B * n / (timing[n]["hamiltonian (eager)"] * 1e-3) / 1e9
    print(f"{n:6d} | {gf:13.1f} | {gf * 1e9 / peak_flops:8.1%} | {gb:12.1f} | {rr:8.2f} | {n / 2:8.0f}")
cross = next((n for n, rr in zip(NS, ratio) if rr > 1), None)
RESULTS["crossover_n_eager"] = cross
print("\n해밀턴(eager)이 밀집층보다 빨라지기 시작하는 n:", cross)

### 7-3. GPU 병렬성: 일이 적으면 GPU는 놀고 있다
너비 $n=1024$로 고정하고 배치 크기 $B$를 1부터 늘려 본다. 시간이 거의 변하지 않는 구간은
GPU의 수천 개 코어가 다 채워지지 않아 **커널 실행 비용(지연 시간)** 이 시간을 결정하는 구간이다.

In [ ]:
n = 1024
dense = nn.Linear(n, n).to(device); ham = HamiltonianLinear(n).to(device); f_ham = ham_plain(ham)
BS = [1, 4, 16, 64, 256, 1024] if QUICK else [1, 4, 16, 64, 256, 1024, 4096, 16384]
bt = {"dense": [], "hamiltonian (eager)": []}
for b in BS:
    x = torch.randn(b, n, device=device)
    bt["dense"].append(bench(lambda: dense(x))); bt["hamiltonian (eager)"].append(bench(lambda: f_ham(x)))
plt.figure(figsize=(6.5, 4))
for k, v in bt.items():
    plt.loglog(BS, v, "o-", label=k)
plt.xlabel("batch size B"); plt.ylabel("forward time (ms)"); plt.title(f"Batch sweep (n={n})"); plt.legend()
savefig("7_batch_sweep")
for b, d, h in zip(BS, bt["dense"], bt["hamiltonian (eager)"]):
    print(f"B={b:6d}: dense {d:.4f} ms, ham {h:.4f} ms, 표본 하나당 dense {d / b * 1e3:.3f} μs")
RESULTS["batch_sweep_n1024"] = {str(b): [d, h] for b, d, h in zip(BS, bt["dense"], bt["hamiltonian (eager)"])}

### 7-4. 커널 수 세기 (프로파일러)
해밀턴 층은 `index_select`, `pad`, 곱하기, 곱하기, 더하기, 더하기처럼 **작은 연산 여러 개**로 이루어져 GPU 커널을 여러 번 띄운다.
밀집층은 cuBLAS 행렬곱 커널 하나다. `torch.compile`은 이 작은 연산들을 커널 하나로 합친다(커널 융합).

In [ ]:
from torch.profiler import profile, ProfilerActivity
if device == "cuda":
    n = 1024; x = torch.randn(512, n, device=device)
    fns = {"dense": lambda: dense(x), "hamiltonian (eager)": lambda: f_ham(x)}
    if TRY_COMPILE:
        fc = torch.compile(f_ham, dynamic=False); fc(x); fns["hamiltonian (compiled)"] = lambda: fc(x)
    kernels = {}
    for name, fn in fns.items():
        with torch.no_grad(), profile(activities=[ProfilerActivity.CUDA]) as prof:
            fn(); torch.cuda.synchronize()
        names = [e.name for e in prof.events() if str(getattr(e, "device_type", "")).endswith("CUDA")]
        kernels[name] = len(names)
        print(f"{name:24s} GPU 커널 {len(names)}개: {[nm[:40] for nm in names]}")
    RESULTS["kernel_count"] = kernels
else:
    print("GPU가 없어 생략")

### 7-5. 학습(순전파+역전파)과 CPU 비교
학습 한 단계에는 역전파가 붙는다(2절에서 유도한 식, 역시 $O(n)$). CPU는 코어 수가 적어 '계산량'이 시간에 더 직접적으로 반영된다.

In [ ]:
def train_step_time(layer, n, B, dev):
    x = torch.randn(B, n, device=dev, requires_grad=True)
    def step():
        with torch.enable_grad():
            y = layer(x); y.sum().backward()
    return bench(step, iters=20, dev=dev)

rows = []
for dev in ([device, "cpu"] if device == "cuda" else ["cpu"]):
    for n in ([256, 1024] if QUICK else [256, 1024, 4096]):
        d = train_step_time(nn.Linear(n, n).to(dev), n, 512, dev)
        h = train_step_time(HamiltonianLinear(n).to(dev), n, 512, dev)
        rows.append((dev, n, d, h)); print(f"{dev:4s} n={n:5d}: 학습 1단계 dense {d:8.3f} ms | ham {h:8.3f} ms | 가속 {d / h:6.2f}배 (FLOP 비 {n // 2})")
RESULTS["train_step_ms"] = [dict(zip(["device", "n", "dense", "ham"], r)) for r in rows]

**해석 포인트 (탐구노트에 기록할 것)**
1. 이론 FLOP 비는 $n/2$배(예: $n=4096$이면 2048배)인데 실제 가속은 몇 배였는가? 왜 그렇게 큰 차이가 나는가?
   → 밀집층은 산술 강도가 높아 GPU 연산 장치를 거의 꽉 채워 쓰고(활용률 표), 해밀턴 층은 메모리 대역폭과 커널 실행 비용에 묶인다.
2. $n$이 작을 때 해밀턴 층이 느리거나 비슷한 이유 → 커널 수(7-4)와 배치 실험(7-3)의 평평한 구간
3. 마스크 밀집 방식은 연결을 줄여도 **전혀 빨라지지 않는다** → '희소하게 만든다 ≠ 빠르다'. 구조를 이용한 구현(gather)이 필요하다.
4. 범용 희소 행렬(CSR)보다 전용 구현이 빠른가? → 구조(행마다 정확히 2개)를 알면 인덱스를 저장·해석할 필요가 없다.
5. `torch.compile`(커널 융합)이 작은 $n$에서 얼마나 도움이 되었나? → 커널 실행 비용 가설의 검증

## 8. 결과 정리와 저장
아래 셀은 이번 실행의 숫자를 모아 `outputs/summary.md`와 `outputs/results.json`으로 저장한다.
이 숫자를 `docs/탐구노트.md`(이 폴더의 `탐구노트.md`)에 옮겨 적고, 각 결과를 **자기 말로** 해석해 둔다.

In [ ]:
lines = [f"# 실행 결과 요약", "", f"- 실행 환경: {GPU_NAME}, PyTorch {torch.__version__}, 데이터 {DATASET}, "
         f"{EPOCHS} epoch, seed {SEEDS}", ""]
lines += ["## 실험 1: 정확도", "", "| 모델 | 정확도(%) | 파라미터 | 에폭당 시간(s) |", "|---|---|---|---|"]
for name, r in exp1.items():
    lines.append(f"| {name} | {100*r['acc_mean']:.2f} ± {100*r['acc_std']:.2f} | {r['params']:,} | {r['epoch_sec']:.2f} |")
e2 = RESULTS["exp2"]
lines += ["", "## 실험 2: 경로 학습", "",
          f"- 장난감 문제 순열 복원율: 순열 {np.mean(RESULTS['toy_recovery']['perm']):.2f}, 해밀턴 {np.mean(RESULTS['toy_recovery']['ham']):.2f}",
          f"- FashionMNIST L=2: 학습(soft) {100*e2['learned_soft']:.2f}%, 경화(hard) {100*e2['learned_hard']:.2f}%, "
          f"경화+미세조정 {100*e2['learned_hard_ft']:.2f}%, "
          f"무작위 순열 {100*e2['random']:.2f}%, P̃ 날카로움 {e2['sharpness']:.3f}"]
lines += ["", f"## 실험 3: 순전파 시간 (B={B}, ms)", "", "| n | " + " | ".join(timing[NS[0]]) + " |",
          "|---" * (len(timing[NS[0]]) + 1) + "|"]
for n in NS:
    lines.append(f"| {n} | " + " | ".join(f"{timing[n].get(k, float('nan')):.4f}" for k in timing[NS[0]]) + " |")
lines += ["", f"- 해밀턴(eager)이 밀집층보다 빨라지는 n: {RESULTS['crossover_n_eager']}"]
if "kernel_count" in RESULTS: lines.append(f"- GPU 커널 수: {RESULTS['kernel_count']}")
open(f"{OUT}/summary.md", "w", encoding="utf-8").write("\n".join(lines))
json.dump(RESULTS, open(f"{OUT}/results.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)
from IPython.display import Markdown, display
display(Markdown("\n".join(lines)))

### GitHub에 저장 (선택)
왼쪽 🔑 **보안 비밀**에 `GITHUB_TOKEN`(저장소 쓰기 권한이 있는 토큰)을 등록했다면, 아래 셀이 이 노트북(실행 결과 포함)과 `outputs/`를 GitHub에 올린다.
실행 날짜가 커밋 기록으로 남으므로 '언제 직접 실행했는지'의 기록이 된다.

In [ ]:
if IN_COLAB and os.path.isfile("save_to_github.py"):
    import importlib, save_to_github; importlib.reload(save_to_github)
    save_to_github.save(message="해밀턴 경로 신경망: Colab 실행 결과 저장")
else:
    print("Colab에서 저장소를 clone한 경우에만 동작합니다.")